# Detecting entanglement dimensionality with isotypic projective measurements

**Companion notebook to the thesis.** The goal is to use *isotypic projective measurements*
(isotypic / Young projectors of the symmetric group) to detect entanglement dimensionality
— Schmidt rank, Schmidt number, and tensor rank — across partitions of multipartite states.

The central object is the isotypic projector $\Pi_\lambda$ on $(\mathbb{C}^d)^{\otimes k}$.
Its key property is
$$\big\|(\Pi_\lambda\otimes I_B)\,|\psi_{AB}\rangle^{\otimes k}\big\|^2 = f^\lambda\, s_\lambda(s_1^2,\dots,s_r^2),$$
where the $s_i$ are the Schmidt coefficients of $|\psi_{AB}\rangle$, $f^\lambda=\dim V^\lambda$,
and $s_\lambda$ is the Schur polynomial. This norm vanishes **iff** the Schmidt rank is below
the height $h(\lambda)$, and the construction extends to mixed states through a
convex-roof / semidefinite-programming relaxation.

The notebook implements and numerically checks each step: characters and isotypic projectors of
$S_k$, the vanishing condition on the computational basis, the closed-form projected norm, the
Schur-polynomial identity, and the Schmidt-number SDP.

> Runs in a **SageMath** kernel; the SDP section additionally needs `picos` and `cvxopt`.

## 1. Mathematical background

### 1.1 Pure states, measurement, tensor products, gates

The smallest unit of quantum information is the **qubit**, a unit vector
$|\psi\rangle\in\mathbb{C}^2$; more generally a **qudit** is a unit vector
$|\psi\rangle\in\mathbb{C}^d$ in a Hilbert space $\mathcal{H}=\mathbb{C}^d$. The conjugate is
$\langle\psi|=(|\psi\rangle)^\dagger$, the inner product of $u,v$ is $\langle u|v\rangle$, and the
projector onto $|\psi\rangle$ is $|\psi\rangle\langle\psi|$.

**Projective measurement.** For a Hermitian operator $O$ with eigenbasis $\{o_i,|v_i\rangle\}_i$
and a state $|\psi\rangle$, the probability of outcome $o_i$ is
$p(o_i)=\langle\psi|v_i\rangle\langle v_i|\psi\rangle$, after which the state collapses to
$|v_i\rangle$.

**Composite systems.** The state space of a composite system is the tensor product of the parts:
for $|\psi\rangle\in\mathbb{C}^n$, $|\phi\rangle\in\mathbb{C}^m$ we have
$|\psi\rangle\otimes|\phi\rangle\in\mathbb{C}^{nm}$. Not every vector in $\mathbb{C}^{nm}$
factorises — the origin of entanglement.

**Gates.** Quantum gates are unitaries $U$ ($U^\dagger U=I$) acting on one or more qudits and
returning pure states.

### 1.2 Entanglement, Schmidt rank and Schmidt number

A bipartite pure state $|\psi_{AB}\rangle\in\mathcal{H}^A\otimes\mathcal{H}^B$ is **separable** if
it factorises and **entangled** otherwise. Every bipartite pure state has a Schmidt decomposition
$$|\psi_{AB}\rangle=\sum_{i=1}^r s_i\,|v_i\rangle_A\otimes|w_i\rangle_B,$$
with orthonormal $\{|v_i\rangle\}$, $\{|w_i\rangle\}$ and $s_i>0$. The minimal $r$ is the **Schmidt
rank** $\mathrm{SR}(|\psi_{AB}\rangle)$ (the rank of the coefficient matrix), and the $s_i$ with
$\sum_i s_i^2=1$ are the **Schmidt coefficients** — the quantities the rest of this notebook works
with.

**Mixed states.** A mixed state is a convex mixture $\rho=\sum_i p_i|\psi_i\rangle\langle\psi_i|$.
The generalisation of Schmidt rank is the **Schmidt number**
$$\mathrm{SN}(\rho)=\min_{\mathcal{D}(\rho)}\ \max_i\ \mathrm{SR}(|\psi_i\rangle),$$
minimised over all decompositions $\mathcal{D}(\rho)=\{p_i,|\psi_i\rangle:\rho=\sum_i p_i|\psi_i\rangle\langle\psi_i|\}$.
Deciding separability — and the Schmidt number more generally — is computationally hard, which
motivates the projector-based detectors below.

**Reduced states.** With orthonormal bases $\{|e^A_i\rangle\}$, $\{|e^B_j\rangle\}$, the partial
trace gives
$$\rho_B=\operatorname{tr}_A(\rho_{AB})=\sum_i(\langle e^A_i|\otimes I_B)\,\rho_{AB}\,(|e^A_i\rangle\otimes I_B).$$

### 1.3 Representation theory essentials

A **representation** of a finite group $G$ on $V$ is a homomorphism $G\to GL(V)$ sending each
element to an invertible matrix while respecting the group law. It is **irreducible** (an *irrep*)
if it has no non-trivial invariant subspace, and every representation of a finite group decomposes
as a direct sum of irreps.

The **group algebra** $A(G)$ is the complex span of the group elements, $a=\sum_{g\in G}a(g)\,g$.
Representations extend linearly and multiplicatively to $A(G)$, with $V(e)=I$ and, for unitary
reps, $V(a^*)=V(a)^\dagger$. **Minimal projections** $p\in A(G)$ ($p^2=p$, indecomposable)
correspond one-to-one with irreducible subspaces; two are either equivalent ($upv=q$) or disjoint
($puq=0$).

**Characters.** The character of $V$ is $\chi(g)=\operatorname{tr}V(g)$; equivalent representations
share a character. With the inner product
$$\langle f,h\rangle_G=\frac1{|G|}\sum_{g\in G}f(g)\,\overline{h(g)},$$
irreducible characters are orthonormal, $\langle\chi_{\lambda_1},\chi_{\lambda_2}\rangle_G=\delta_{\lambda_1\lambda_2}$.
Characters are constant on conjugacy classes — the fact that makes the projector formula below
efficient.

## 2. The symmetric group, Young diagrams and isotypic projectors

### The group $S_k$ and its commutant

Let $S_k$ act on $(\mathbb{C}^d)^{\otimes k}$ by permuting the tensor factors (in cycle notation
$\pi_{(a\,b\,c\,d)}$ rotates the indicated factors). Independently, $U\in U(d)$ acts diagonally,
$U:|e_{i_1}\rangle\otimes\cdots\otimes|e_{i_k}\rangle\mapsto U|e_{i_1}\rangle\otimes\cdots\otimes U|e_{i_k}\rangle$.
On every basis vector the two actions commute, hence $S_k$ and $U(d)$ commute on all of
$(\mathbb{C}^d)^{\otimes k}$ — the starting point of Schur–Weyl duality.

### Young diagrams, tableaux and symmetrizers

A **partition** $\lambda\vdash k$ (non-increasing, summing to $k$) is drawn as a **Young diagram**
with $\lambda_i$ boxes in row $i$. A **Young tableau** fills the boxes; a **standard** tableau (SYT)
uses each of $1,\dots,k$ once, increasing along rows and down columns. The number of SYT of shape
$\lambda$ is the hook-length formula
$$f^\lambda=\frac{k!}{\prod_{\square\in\lambda}h(\square)}.$$
To a tableau $T$ one associates the row- and column-symmetrizers
$$r(T)=\sum_{\pi\in R(T)}\pi,\qquad c(T)=\sum_{\pi\in C(T)}\operatorname{sign}(\pi)\,\pi,$$
and the **Young symmetrizer** $e_T=r(T)\,c(T)$.

> **Theorem (Christandl 1.14).** For a standard tableau $T$ of shape $\lambda$,
> $\tfrac{f^\lambda}{k!}e_T$ is a minimal projection onto the irrep $V^\lambda$ of $S_k$, with
> $\dim V^\lambda=f^\lambda$. The $V^\lambda$ for $\lambda\vdash k$ form a complete set of irreps,
> and $\;\mathbb{C}[S_k]\cong\bigoplus_{\lambda\vdash k}(V^\lambda)^{\oplus f^\lambda}.$

### The isotypic projector

Summing the minimal projections within one isotypic component gives the **isotypic projector** onto
the $\lambda$-sector,
$$\Pi_\lambda=\frac{\chi_\lambda(\mathrm{id})}{k!}\sum_{\sigma\in S_k}\chi_\lambda(\sigma^{-1})\,\sigma=\frac{f^\lambda}{k!}\sum_{\sigma\in S_k}\chi_\lambda(\sigma)\,\sigma.$$
Because $\chi_\lambda$ is constant on conjugacy classes, the code evaluates it once per class rather
than once per group element.

In [15]:
import math
import itertools
from itertools import product as iproduct

import numpy as np

import sage.all as sage
from sage.rings.rational_field import QQ
from sage.combinat.sf.sf import SymmetricFunctions

# Ring of symmetric functions over the rationals, with the two bases we need.
_Sym = SymmetricFunctions(QQ)
_s   = _Sym.schur()       # Schur basis      s_lambda
_p   = _Sym.powersum()    # power-sum basis  p_lambda


### 2.1 Characters of $S_k$ on its conjugacy classes

We first extract the irreducible character $\chi_\lambda$ from Sage's character table, evaluated on
every conjugacy class — the data that enters the projector formula.

In [16]:
def character_on_conjugacy_classes(k: int, lam: list[int]) -> list[tuple[list, object]]:
    """Irreducible character chi_lambda of S_k evaluated on every conjugacy class.

    Parameters
    ----------
    k   : size of the symmetric group S_k.
    lam : partition of k labelling the irreducible representation V^lambda.

    Returns
    -------
    A list of (class_elements, chi_value) pairs, one per conjugacy class:
    ``class_elements`` lists the permutations in the class and ``chi_value`` is
    chi_lambda evaluated there (constant on the class).
    """
    assert sum(lam) == k
    lam = sorted(lam, reverse=True)              # normalise to a non-increasing partition

    Sk = sage.SymmetricGroup(k)
    char_table = Sk.character_table()            # rows: irreps, columns: conjugacy classes
    classes = Sk.conjugacy_classes()

    # Sage orders character-table rows by partitions in reverse-lexicographic order.
    partitions_ordered = list(reversed(sage.Partitions(k).list()))
    row_idx = partitions_ordered.index(lam)

    return [(cl.list(), char_table[row_idx][j]) for j, cl in enumerate(classes)]


# Sanity check: reproduce the known character table of S_3.
for lam in sage.Partitions(3).list():
    result = character_on_conjugacy_classes(3, lam)
    print(f"\nlambda = {lam}:")
    for elements, chi in result:
        print(f"  sigma = {elements[0]}, chi = {chi}")



lambda = [3]:
  sigma = (), chi = 1
  sigma = (2,3), chi = 1
  sigma = (1,2,3), chi = 1

lambda = [2, 1]:
  sigma = (), chi = 2
  sigma = (2,3), chi = 0
  sigma = (1,2,3), chi = -1

lambda = [1, 1, 1]:
  sigma = (), chi = 1
  sigma = (2,3), chi = -1
  sigma = (1,2,3), chi = 1


### 2.2 The permutation action on $(\mathbb{C}^n)^{\otimes k}$

Each $\sigma\in S_k$ is represented as an $n^k\times n^k$ permutation matrix reordering the tensor
factors. We build it column by column on the computational basis.

In [17]:
def permutation_matrix_on_tensor_power(sigma, n: int, k: int) -> np.ndarray:
    """Matrix of a permutation sigma in S_k acting on (C^n)^{otimes k}.

    The symmetric group permutes the k tensor factors:
        sigma |e_{i_1}> (x) ... (x) |e_{i_k}>  =  |e_{i_{sigma^{-1}(1)}}> (x) ... .
    The result is an (n^k) x (n^k) permutation matrix (a single 1 per column).

    Parameters
    ----------
    sigma : element of Sage's SymmetricGroup(k).
    n     : local dimension (single-factor space V = C^n).
    k     : number of tensor factors.
    """
    basis = list(iproduct(range(n), repeat=k))     # all index tuples (i_1, ..., i_k)
    index = {b: i for i, b in enumerate(basis)}    # tuple -> row/column position

    sigma_inv = sigma.inverse()
    dim = n ** k
    M = np.zeros((dim, dim), dtype=complex)

    for col_idx, basis_vec in enumerate(basis):
        # Output factor j reads input factor sigma^{-1}(j) (Sage is 1-indexed).
        new_basis = tuple(basis_vec[sigma_inv(j + 1) - 1] for j in range(k))
        row_idx = index[new_basis]
        M[row_idx, col_idx] = 1.0

    return M


### 2.3 Building $\Pi_\lambda$

We assemble $\Pi_\lambda=\frac{f^\lambda}{k!}\sum_\sigma\chi_\lambda(\sigma)\,\sigma$, looping over
conjugacy classes (constant character) and summing the permutation matrices.

In [18]:
def isotypic_projector(lam: list[int], n: int, k: int) -> np.ndarray:
    """Isotypic projector Pi_lambda onto the lambda-component of (C^n)^{otimes k}.

    Implements
        Pi_lambda = (f^lambda / k!) * sum_{sigma in S_k} chi_lambda(sigma) sigma,
    where f^lambda = chi_lambda(id) = dim V^lambda. Since chi_lambda is constant
    on conjugacy classes, we read it once per class and reuse it for all members.

    Parameters
    ----------
    lam : partition of k labelling the target irrep V^lambda.
    n   : local dimension.
    k   : number of tensor factors (must equal sum(lam)).
    """
    assert sum(lam) == k
    lam = sorted(lam, reverse=True)

    Sk = sage.SymmetricGroup(k)
    char_table = Sk.character_table()
    classes = Sk.conjugacy_classes()

    # Locate the character-table row for lambda (reverse-lex ordering, as above).
    partitions_ordered = list(reversed(sage.Partitions(k).list()))
    row_idx = partitions_ordered.index(lam)

    # f^lambda = chi_lambda(identity) is the dimension of the irrep.
    id_idx = next(j for j, cl in enumerate(classes)
                  if cl.representative().is_one())
    d_lam = int(char_table[row_idx][id_idx])

    dim = n ** k
    Pi = np.zeros((dim, dim), dtype=complex)

    # Accumulate chi_lambda(sigma) * sigma over the whole group, class by class.
    for j, cl in enumerate(classes):
        chi = complex(char_table[row_idx][j])      # character is constant on the class
        for sigma in cl:
            M = permutation_matrix_on_tensor_power(sigma, n, k)
            Pi += chi * M

    Pi *= d_lam / sage.factorial(k)                # overall prefactor f^lambda / k!
    return Pi


### 2.4 Sanity checks

A valid isotypic projector must be Hermitian and idempotent ($\Pi_\lambda^2=\Pi_\lambda$), have
integer trace $f^\lambda\times(\text{multiplicity})$, and the projectors over all $\lambda\vdash k$
must resolve the identity.

In [19]:
n, k = 2, 3  # qubit example: V = C^2 with k = 3 tensor factors

for lam in sage.Partitions(k).list():
    Pi = isotypic_projector(lam, n, k)

    # A projector must be idempotent ...
    assert np.allclose(Pi @ Pi, Pi), f"{lam}: not idempotent"
    # ... and Hermitian.
    assert np.allclose(Pi, Pi.conj().T), f"{lam}: not Hermitian"

    # trace(Pi_lambda) = f^lambda * (multiplicity of V^lambda in the tensor space).
    print(f"lambda={lam}, trace={np.trace(Pi).real:.1f}")

# The isotypic projectors over all lambda |- k resolve the identity.
total = sum(isotypic_projector(lam, n, k)
            for lam in sage.Partitions(k).list())
assert np.allclose(total, np.eye(n**k))
print("Projectors sum to identity \u2713")


lambda=[3], trace=4.0
lambda=[2, 1], trace=4.0
lambda=[1, 1, 1], trace=0.0
Projectors sum to identity ✓


## 3. Computational basis vectors and the vanishing condition

### The occupation type $\nu$

Fix the standard basis $\{|e_i\rangle\}$ of $\mathbb{C}^d$. A basis vector
$|e_{i_1}\rangle\otimes\cdots\otimes|e_{i_k}\rangle$ is summarised by its **occupation type**
$\nu=(\nu_1,\dots,\nu_{h(\nu)})$: the sorted label multiplicities, $\nu_1$ the count of the most
frequent label. The number of distinct labels $h(\nu)$ is the **height** of $\nu$.

### Vanishing of $\Pi_\lambda$ on basis vectors

Say $\lambda$ **majorises** $\nu$ when $\sum_{i\le q}\lambda_i\ge\sum_{i\le q}\nu_i$ for all $q$. If
$\lambda$ does **not** majorise $\nu$ — which necessarily holds when $h(\nu)<h(\lambda)$ — then the
basis vector is annihilated,
$$\Pi_\lambda\big(|e_{i_1}\rangle\otimes\cdots\otimes|e_{i_k}\rangle\big)=0.$$
A sector $\lambda$ can only be populated when the state has enough distinct components (Christandl's
vanishing condition).

### 3.1 Basis kets

Helper to materialise a computational basis vector $|i_1,\dots,i_k\rangle$ as an explicit vector in
$(\mathbb{C}^n)^{\otimes k}\cong\mathbb{C}^{n^k}$.

In [20]:
import numpy as np
from itertools import product as iproduct

def ket(indices: list[int], n: int) -> np.ndarray:
    """
    Convert a basis vector in ket notation to a vector in (C^n)^{otimes k}.
    
    indices: tuple of ints, e.g. (2, 0, 3) for |2,0,3>
    n: local dimension, e.g. n=4 for C^4
    
    Example: ket((2,0,3), n=4) -> unit vector in C^{4^3} = C^64
    """
    k = len(indices)
    assert all(0 <= i < n for i in indices), f"All indices must be in range [0, {n-1}]"

    basis = list(iproduct(range(n), repeat=k))
    index = {b: i for i, b in enumerate(basis)}

    v = np.zeros(n**k, dtype=complex)
    v[index[tuple(indices)]] = 1.0
    return v

# |2,0,3> in (C^4)^{otimes 3}
v = ket([2,0,3], n=4)
print(v.shape)   # (64,)
print(v.sum())   # 1.0 — exactly one nonzero entry

# Recover the index
print(np.argmax(v))  # 2*16 + 0*4 + 3 = 35

(64,)
(1+0j)
35


### 3.2 Projecting a ket and the orthogonality of sectors

We project a basis vector onto a chosen sector and verify idempotence on the image, plus that the
symmetric, mixed and antisymmetric components are mutually orthogonal and reconstruct the original
vector.

In [21]:
v = ket([0, 0, 3], n=4)
Pi = isotypic_projector([3], n=4, k=3)

projected = Pi @ v

# Norm of the projected vector — how much of |2,0,3> lives in this subspace
print(np.linalg.norm(projected))

# Applying the projector twice should give the same result
print(np.allclose(Pi @ projected, projected))  # True

# Projectors onto different sectors are orthogonal
Pi_sym  = isotypic_projector([3],     n=4, k=3)
Pi_mix  = isotypic_projector([2,1],   n=4, k=3)
Pi_anti = isotypic_projector([1,1,1], n=4, k=3)

v_sym  = Pi_sym  @ v
v_mix  = Pi_mix  @ v
v_anti = Pi_anti @ v

print(v_anti)

# These three components are orthogonal and reconstruct v
print(np.allclose(v_sym + v_mix + v_anti, v))           # True
print(np.allclose(np.dot(v_sym.conj(), v_mix), 0))      # True

0.5773502691896257
True
[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
True
True


### 3.3 The occupation type and the direct vanishing test

`occupation_type(v, n)` returns the type $\nu$. `projected_norm(lam, n, v)` builds $\Pi_\lambda$
explicitly and returns the number $\lVert\Pi_\lambda|v\rangle\rVert$, and `projects_to_zero` is just
the boolean predicate "that norm is below tolerance". The test below prints, for a fixed state, each
sector's height, its **actual** projected norm, whether it is zero, and the **expected** outcome,
with a PASS/FAIL flag. (The sufficient rule shown here is $h(\nu)<h(\lambda)\Rightarrow$ vanish; the
exact iff criterion follows in 3.4.)

In [22]:
def occupation_type(v: list[int], n: int) -> list[int]:
    """Occupation type nu of a basis vector |e_{v_1}, ..., e_{v_k}>.

    Counts how often each of the n local labels occurs in ``v`` and returns
    those multiplicities sorted non-increasingly, with zeros dropped.
    Example: occupation_type([0, 0, 1], n=2) -> [2, 1].
    """
    counts = [0] * n              # n could equivalently be inferred as max(v) + 1
    for label in v:
        counts[label] += 1
    counts.sort(reverse=True)
    return [c for c in counts if c != 0]


def projected_norm(lam: list[int], n: int, v: list[int]) -> float:
    """|| Pi_lambda |v> ||  for a single computational basis vector |v>.

    Builds the isotypic projector explicitly and applies it to |v>.
    """
    k = sum(lam)
    Pi = isotypic_projector(lam, n, k)
    return float(np.linalg.norm(Pi @ ket(v, n)))


def projects_to_zero(lam: list[int], n: int, v: list[int], tol: float = 1e-5) -> bool:
    """True iff Pi_lambda annihilates the basis vector |v> (projected norm < tol)."""
    return projected_norm(lam, n, v) < tol


In [23]:
def run_vanishing_tests(n: int, v: list[int],
                        cases: list[tuple[list[int], bool]]) -> None:
    """Pretty-print a vanishing test for a fixed basis vector |v>.

    cases : list of (lambda, expected_to_vanish) pairs.
    For each sector lambda it shows the height h(lambda), the actual projected
    norm || Pi_lambda |v> ||, whether that is zero, the expected outcome, and a
    PASS/FAIL flag.
    """
    nu = occupation_type(v, n)
    k = len(v)
    print(f"State        |v> = |{','.join(map(str, v))}>   in (C^{n})^(x){k}")
    print(f"Occupation   nu  = {nu}   (height h(nu) = {len(nu)})")
    print(f"Vanishes when lambda cannot dominate nu  (always if h(lambda) > h(nu))\n")

    print(f"  {'lambda':<16}{'h(lam)':>6}   {'||Pi|v>||':>10}   {'got':>8}   {'expected':>8}   status")
    print("  " + "-" * 64)
    for lam, expect_zero in cases:
        norm = projected_norm(lam, n, v)
        is_zero  = norm < 1e-5
        got      = "ZERO" if is_zero else "nonzero"
        expected = "ZERO" if expect_zero else "nonzero"
        status   = "PASS" if is_zero == expect_zero else "FAIL  <<<"
        print(f"  {str(lam):<16}{len(lam):>6}   {norm:>10.4f}   "
              f"{got:>8}   {expected:>8}   {status}")


# (C^2)^(x)3 : only two distinct labels are available, so any sector whose
# height exceeds 2 must vanish.
run_vanishing_tests(
    n=2,
    v=[0, 0, 1],
    cases=[
        ([3],       False),   # fully symmetric          -> populated
        ([2, 1],    False),   # mixed symmetry           -> populated
        ([1, 1, 1], True),    # antisymmetric needs 3 distinct labels -> vanishes
    ],
)


State        |v> = |0,0,1>   in (C^2)^(x)3
Occupation   nu  = [2, 1]   (height h(nu) = 2)
Vanishes when lambda cannot dominate nu  (always if h(lambda) > h(nu))

  lambda          h(lam)    ||Pi|v>||        got   expected   status
  ----------------------------------------------------------------
  [3]                  1       0.5774    nonzero    nonzero   PASS
  [2, 1]               2       0.8165    nonzero    nonzero   PASS
  [1, 1, 1]            3       0.0000       ZERO       ZERO   PASS


In [ ]:
# (C^3)^(x)6 with v = [0,0,0,1,1,2] : occupation type nu = [3,2,1], so h(nu) = 3.
# Sectors of height <= 3 can be populated; height 4 and 5 must vanish.
# run_vanishing_tests(
#     n=3,
#     v=[0, 0, 0, 1, 1, 2],
#     cases=[
#         ([6],             False),   # h(lam) = 1
#         ([4, 2],          False),   # h(lam) = 2
#         ([3, 3],          False),   # h(lam) = 2
#         ([3, 2, 1],       False),   # h(lam) = 3 = h(nu)
#         ([2, 2, 1, 1],    True),    # h(lam) = 4 > h(nu) = 3  -> vanishes
#         ([2, 1, 1, 1, 1], True),    # h(lam) = 5 > h(nu) = 3  -> vanishes
#     ],
# )


State        |v> = |0,0,0,1,1,2>   in (C^3)^(x)6
Occupation   nu  = [3, 2, 1]   (height h(nu) = 3)
Vanishes when lambda cannot dominate nu  (always if h(lambda) > h(nu))

  lambda          h(lam)    ||Pi|v>||        got   expected   status
  ----------------------------------------------------------------
  [6]                  1       0.1291    nonzero    nonzero   PASS
  [4, 2]               2       0.5477    nonzero    nonzero   PASS
  [3, 3]               2       0.2887    nonzero    nonzero   PASS
  [3, 2, 1]            3       0.5164    nonzero    nonzero   PASS
  [2, 2, 1, 1]         4       0.0000       ZERO       ZERO   PASS
  [2, 1, 1, 1, 1]      5       0.0000       ZERO       ZERO   PASS


### 3.4 Majorisation form of the condition

The exact criterion is: $\Pi_\lambda$ annihilates $|v\rangle$ **iff** $\lambda$ does *not* majorise
$\nu$. `majorizes(lam, nu)` implements the prefix-sum test ($\lambda$ majorises $\nu$ when every
cumulative sum of $\lambda$ is $\ge$ that of $\nu$) and `cumulative_sums` exposes those prefix sums.
The test prints both prefix-sum vectors, the majorisation verdict, the prediction it implies, and the
actual projected norm.

In [25]:
def cumulative_sums(p: list[int]) -> list[int]:
    """Prefix sums [p_1, p_1+p_2, ...]; the data compared in the majorisation test."""
    out, running = [], 0
    for x in p:
        running += x
        out.append(running)
    return out


def majorizes(lam: list[int], nu: list[int]) -> bool:
    """True iff lambda majorises nu: every prefix sum of lambda is >= that of nu.

    Both partitions sum to the same k; the shorter prefix-sum vector is padded
    with k so the two have equal length before the element-wise comparison.
    """
    assert sum(lam) == sum(nu)
    k = sum(lam)
    lam_c = cumulative_sums(lam)
    nu_c  = cumulative_sums(nu)
    length = max(len(lam_c), len(nu_c))
    lam_c += [k] * (length - len(lam_c))
    nu_c  += [k] * (length - len(nu_c))
    return all(a >= b for a, b in zip(lam_c, nu_c))


In [26]:
# Exact criterion:  Pi_lambda |v> = 0  <=>  lambda does NOT majorize nu.
# For each sector we print the prefix sums of lambda and nu, the majorization
# verdict, the prediction it implies, and the actual projected norm.
# n = 3
# v = [0, 0, 0, 1, 1, 2]
# nu = occupation_type(v, n)

# print(f"State |v> = |{','.join(map(str, v))}>   nu = {nu}   prefix(nu) = {cumulative_sums(nu)}\n")
# print(f"  {'lambda':<16}{'prefix(lam)':<18}{'maj?':>5}   {'predicted':>9}   {'||Pi|v>||':>10}   status")
# print("  " + "-" * 72)

# for lam in [[6], [4, 2], [3, 3], [3, 2, 1], [2, 2, 1, 1], [2, 1, 1, 1, 1]]:
#     maj = majorizes(lam, nu)               # does lambda majorize nu?
#     predicted_zero = not maj               # it vanishes exactly when it does not
#     norm = projected_norm(lam, n, v)
#     actual_zero = norm < 1e-5
#     status = "PASS" if predicted_zero == actual_zero else "FAIL  <<<"
#     print(f"  {str(lam):<16}{str(cumulative_sums(lam)):<18}"
#           f"{('yes' if maj else 'no'):>5}   "
#           f"{('ZERO' if predicted_zero else 'nonzero'):>9}   {norm:>10.4f}   {status}")


State |v> = |0,0,0,1,1,2>   nu = [3, 2, 1]   prefix(nu) = [3, 5, 6]

  lambda          prefix(lam)        maj?   predicted    ||Pi|v>||   status
  ------------------------------------------------------------------------
  [6]             [6]                 yes     nonzero       0.1291   PASS
  [4, 2]          [4, 6]              yes     nonzero       0.5477   PASS
  [3, 3]          [3, 6]              yes     nonzero       0.2887   PASS
  [3, 2, 1]       [3, 5, 6]           yes     nonzero       0.5164   PASS
  [2, 2, 1, 1]    [2, 4, 5, 6]         no        ZERO       0.0000   PASS
  [2, 1, 1, 1, 1] [2, 3, 4, 5, 6]      no        ZERO       0.0000   PASS


## 4. The projected norm in terms of Schmidt coefficients

For a basis vector, $\langle v_\nu|\Pi_\lambda|v_\nu\rangle$ depends only on the occupation type
$\nu$. Using $\Pi_\lambda^\dagger\Pi_\lambda=\Pi_\lambda$, Frobenius reciprocity, and Young's rule
$M^\nu\cong\bigoplus_\lambda K_{\lambda\nu}V^\lambda$ (with $K_{\lambda\nu}$ the Kostka numbers),
$$\|\Pi_\lambda|v_\nu\rangle\|^2=\langle v_\nu|\Pi_\lambda|v_\nu\rangle=\frac{f^\lambda K_{\lambda\nu}}{\binom{k}{\nu_1,\dots,\nu_r}},$$
where $K_{\lambda\nu}=0$ exactly when $\nu\succ\lambda$ (e.g. when $h(\nu)<h(\lambda)$).

For a general entangled state $|\psi_{AB}\rangle=\sum_{i=1}^r s_i|a_i\rangle_A|b_i\rangle_B$ with
orthonormal $\{|a_i\rangle\}$, summing over occupation types and weighting by the squared Schmidt
coefficients gives
$$\|(\Pi_\lambda\otimes I)\,|\psi_{AB}\rangle^{\otimes k}\|^2=f^\lambda\sum_{\nu\vdash k}K_{\lambda\nu}\,m_\nu(s_1^2,\dots,s_r^2),$$
with $m_\nu$ the monomial symmetric polynomial. In particular this norm is $0$ **iff**
$\mathrm{SR}(|\psi_{AB}\rangle)<h(\lambda)$.

In Section 5 the Kostka sum $\sum_\nu K_{\lambda\nu}\,m_\nu$ is identified as a Schur polynomial,
giving a single closed form valid for every $\lambda$.

## 5. The Schur-polynomial formula

The sum $\sum_{\nu\vdash k}K_{\lambda\nu}\,m_\nu(s_1^2,\dots,s_r^2)$ is exactly the **Schur
polynomial** $s_\lambda$ at the squared Schmidt coefficients. Hence the projected norm has the
compact closed form
$$\boxed{\ \|(\Pi_\lambda\otimes I)\,|\psi_{AB}\rangle^{\otimes k}\|^2=f^\lambda\, s_\lambda(s_1^2,s_2^2,\dots,s_r^2)\ }$$
valid for **every** partition $\lambda$. Since $s_\lambda$ vanishes when its number of variables $r$
falls below $h(\lambda)$, this recovers the criterion $\|\cdot\|^2=0\iff\mathrm{SR}<h(\lambda)$ while
the numerical value encodes the whole Schmidt spectrum. Below we compute $f^\lambda$ via the
hook-length formula and $s_\lambda$ via Sage, then check the identity against the explicit
projector.

In [27]:
def f_lambda(lam: list[int]) -> int:
    """Number of standard Young tableaux of shape lam, via the hook-length formula f^lambda = k! / prod h(u)."""
    lam = sorted(lam, reverse=True)
    k = sum(lam)
    hook_product = 1
    for i, row_len in enumerate(lam):
        for j in range(row_len):
            arm = row_len - j - 1                          # cells to the right
            leg = sum(1 for r in lam[i + 1:] if r > j)    # cells below
            hook_product *= arm + leg + 1
    return math.factorial(k) // hook_product

# Sanity checks against known values
assert f_lambda([3])     == 1   # single row
assert f_lambda([1,1,1]) == 1   # single column
assert f_lambda([2,1])   == 2
assert f_lambda([3,2,1]) == 16
print("f_lambda checks passed ✓")

f_lambda checks passed ✓


In [28]:
def norm_schur(lam: list[int], sis: list[float]) -> float:
    """||Pi_lambda psi_AB^{otimes k}||^2 = f^lambda * s_lambda(s1^2, ..., sr^2).

    lam: Young frame partition (any order, will be sorted)
    sis: Schmidt coefficients with sum(si^2) = 1
    """
    sisq = [si**2 for si in sis]
    return f_lambda(lam) * schur_polynomial(lam, sisq)


# Quick checks for schur_polynomial
# s_{[1]}(x) = x_1 + x_2  →  sum = 1
# Patch: allow Python floats in evaluation by working over RDF
def schur_polynomial(lam: list[int], x: list[float]) -> float:
    lam = sorted(lam, reverse=True)
    r = len(x)
    poly = _s[lam].expand(r).change_ring(sage.RDF)
    return float(poly(*[sage.RDF(xi) for xi in x]))

assert abs(schur_polynomial([1], [0.3, 0.7]) - 1.0) < 1e-10

# s_{[2]} = h_2: 0.25 + 0.25 + 0.25 = 0.75 at (0.5, 0.5)
assert abs(schur_polynomial([2], [0.5, 0.5]) - 0.75) < 1e-10

# s_{[1,1]} = e_2: 0.5 * 0.5 = 0.25 at (0.5, 0.5)
assert abs(schur_polynomial([1, 1], [0.5, 0.5]) - 0.25) < 1e-10

# height > r  →  0
assert abs(schur_polynomial([1, 1, 1], [0.5, 0.5])) < 1e-10

print("schur_polynomial checks passed ✓")

schur_polynomial checks passed ✓


In [29]:
def norm_sq_projected_general(lam: list[int], sis: list[float]) -> float:
    """Ground-truth ||Pi_lam psi_AB^{otimes k}||^2 for ANY partition lam,
    obtained by building Pi_lam explicitly and summing its weighted diagonal."""
    r = len(sis)
    sisq = [si ** 2 for si in sis]
    k = sum(lam)
    Pi = isotypic_projector(lam, n=r, k=k)
    basis = list(iproduct(range(r), repeat=k))
    basis_index = {b: i for i, b in enumerate(basis)}
    total = 0.0
    for idx_tuple in basis:
        weight = math.prod(sisq[i] for i in idx_tuple)
        if weight == 0:
            continue
        total += weight * Pi[basis_index[idx_tuple], basis_index[idx_tuple]].real
    return total


def assert_close(a: float, b: float, tol: float = 1e-6, label: str = "") -> None:
    err = abs(a - b)
    status = "PASS" if err < tol else "FAIL"
    print(f"[{status}] {label}")
    if err >= tol:
        print(f"        got {a:.8f}, expected {b:.8f}, diff {err:.2e}")


# --- Test 1: norm_schur vs explicit projector, across partition shapes ---
# single row [4], single column [1,1,1,1], and mixed shapes in between.
sis = [0.7, 0.5, 0.3, 0.1]
sis = [s / math.sqrt(sum(x ** 2 for x in sis)) for s in sis]   # normalise sum(s_i^2)=1
k = 4
print(f"norm_schur vs explicit projector (k={k}, r={len(sis)}):")
for lam in [[4], [3, 1], [2, 2], [2, 1, 1], [1, 1, 1, 1]]:
    assert_close(norm_schur(lam, sis), norm_sq_projected_general(lam, sis), label=f"lam={lam}")

# --- Test 2: the f^lam s_lam values over all lam |- k sum to ||psi||^2 = 1 ---
k = 3
sis3 = [1 / math.sqrt(2), 1 / math.sqrt(3), 1 / math.sqrt(6)]
total = sum(norm_schur(list(lam), sis3) for lam in sage.Partitions(k).list())
assert_close(total, 1.0, label=f"sum over all partitions of {k} equals 1")


norm_schur vs explicit projector (k=4, r=4):
[PASS] lam=[4]
[PASS] lam=[3, 1]
[PASS] lam=[2, 2]
[PASS] lam=[2, 1, 1]
[PASS] lam=[1, 1, 1, 1]
[PASS] sum over all partitions of 3 equals 1


## 6. From pure states to mixed states: Schmidt number and the convex roof

Reference: Tóth, Moroder, Gühne, *Phys. Rev. Lett.* **114**, 160501 (2015).

Writing $\Pi^\lambda$ for the isotypic projector, the pure-state result gives
$$\|(\Pi^\lambda_{A_1\dots A_k}\otimes I_{B_1\dots B_k})\,|\psi_{AB}\rangle^{\otimes k}\|^2=0\iff\mathrm{SR}(|\psi_{AB}\rangle)<h(\lambda).$$
For mixed $\rho=\sum_i p_i|\psi_i\rangle\langle\psi_i|$, set
$\omega_{1\dots k}=\sum_i p_i(|\psi_i\rangle\langle\psi_i|)^{\otimes k}$. Then
$$\sum_i p_i\,\|(\Pi^\lambda\otimes I)|\psi_i\rangle^{\otimes k}\|^2=\operatorname{tr}\!\big[(\Pi^\lambda\otimes I)\,\omega_{1\dots k}\big],$$
and minimising over decompositions gives the exact convex-roof statement
$$\min_{\{p_i,\psi_i\}}\operatorname{tr}\!\big[(\Pi^\lambda\otimes I)\,\omega_{1\dots k}\big]=0\iff\mathrm{SN}(\rho)<h(\lambda).$$

### 6.1 SDP relaxation (symmetric extension)

The set of genuine $\omega=\sum_i p_i(|\psi_i\rangle\langle\psi_i|)^{\otimes k}$ is intractable, so we
relax it: $\omega\succeq0$, supported on the symmetric subspace, with the correct one-copy marginal
$\operatorname{tr}_{\neq1}\omega=\rho$, and PPT across the copy cuts. The relaxed set is a superset,
so the SDP optimum **lower-bounds** the convex roof:
$$\min_{\omega}\ \operatorname{tr}\!\big[(\Pi^\lambda_A\otimes I_B)\,\omega\big]\quad\text{s.t.}\quad\omega\succeq0,\ P_{\mathrm{sym}}\,\omega\,P_{\mathrm{sym}}=\omega,\ \operatorname{tr}_{\neq1}\omega=\rho,\ \omega^{T_S}\succeq0.$$
A strictly positive optimum certifies $\mathrm{SN}(\rho)\ge h(\lambda)$; a zero is inconclusive.

## Schmidt-number SDP (symmetric-extension relaxation)

The pure-state result above, $ | (\Pi^\lambda_A\otimes I_B) | \psi_{AB} \rangle^{ \otimes k} | ^2=f^\lambda s_\lambda(s_1^2,\dots,s_r^2)$,
vanishes **iff** $\mathrm{SR}(\psi)<h(\lambda)$. Extending to mixed $\rho$ via the convex roof gives, exactly,

$$\mathrm{SN}(\rho)<h(\lambda)\;\iff\;\min_{\omega}\ \mathrm{tr}\!\big[(\Pi^\lambda_A\otimes I_B)\,\omega\big]=0,$$

the minimum over $\omega=\sum_i p_i(|\psi_i\rangle\langle\psi_i|)^{\otimes k}$ with $\mathrm{tr}_{2\dots k}\omega=\rho$.
That set (convex hull of identical pure powers) is intractable, so we **relax** it to the
Tóth–Moroder–Gühne symmetric extension: $\omega\succeq0$, supported on $\mathrm{Sym}^k$,
with the right one-copy marginal, and PPT across the copy cuts. The relaxed set is a *superset*,
so the SDP optimum **lower-bounds** the convex roof:

$$\boxed{\;v^\star_\lambda>0\ \Longrightarrow\ \mathrm{SN}(\rho)\ge h(\lambda)\;}\qquad(v^\star_\lambda=0:\ \text{inconclusive}).$$

We work in the ordering $A_1\cdots A_k\,B_1\cdots B_k$, so the objective is just
$W=\Pi^\lambda_A\otimes I_{B^{\otimes k}}$ (a Kronecker product), while marginal/PPT act on the
$2k$ subsystems $[\,d_A,\dots,d_A,d_B,\dots,d_B\,]$. A copy is one $AB$ pair; a copy-permutation
acts *simultaneously* on the matching $A_j$ and $B_j$, so $V_\pi=P^A_\pi\otimes P^B_\pi$ and
$P_{\mathrm{sym}}=\frac1{k!}\sum_\pi V_\pi$.

## 7. Compressing the SDP to the symmetric subspace

The variables and constraints of the Schmidt-number SDP live on $(\mathbb C^d)^{\otimes k}$
with $d=d_Ad_B$ (one *copy* is one $A_jB_j$ pair). Since the feasible $\omega$ is supported on the
permutation-symmetric subspace, we can carry it as the much smaller matrix
$\omega_{\mathrm{sym}}=V\,\omega\,V^\dagger$ on $\mathrm{Sym}^k(\mathbb C^d)$, of dimension
$\binom{d+k-1}{k}\ll d^k$. This section collects the closed forms derived in the thesis and
implements each as a method with its own sanity check.

**Isometry to the symmetric subspace.** In the occupation ("Dicke") basis $\{\lvert\nu\rangle\}$,
$\nu\in\mathbb N^d,\ \lvert\nu\rvert=k$,
$$V=\sum_{i_1\dots i_k}\frac{1}{\sqrt{\binom{k}{\nu(i_1\dots i_k)}}}\,\lvert\nu(i_1\dots i_k)\rangle\langle i_1\dots i_k\rvert,
\qquad VV^\dagger=\mathbb I_{\mathrm{sym}},\quad V^\dagger V=\Pi_{\mathrm{sym}}.$$

**Objective.** With $M=\Pi^\lambda_{A_{1\dots k}}\otimes\mathbb I_{B_{1\dots k}}$ and cyclicity,
$$\operatorname{tr}(M\,\omega)=\operatorname{tr}\!\big(\underbrace{V M V^\dagger}_{=:W_{\mathrm{obj}}}\,\omega_{\mathrm{sym}}\big),$$
a constant matrix $W_{\mathrm{obj}}$ on $\mathrm{Sym}^k$ precomputed once.

**One-copy marginal.** Define ladder operators on the Dicke basis,
$a_j^\dagger\lvert\nu\rangle=\sqrt{\nu_j+1}\,\lvert\nu+e_j\rangle$ (so $a_j\lvert\nu\rangle=\sqrt{\nu_j}\,\lvert\nu-e_j\rangle$). Then
$$\operatorname{tr}_{\neq1}(\omega)_{j j'}=\frac1k\,\operatorname{tr}\!\big(a_{j'}^\dagger a_j\,\omega_{\mathrm{sym}}\big),$$
i.e. the single-copy marginal is $\tfrac1k$ times the one-body reduced density matrix
$\gamma_{jj'}=\langle a_{j'}^\dagger a_j\rangle$. The constraint $\operatorname{tr}_{\neq1}\omega=\rho$ becomes
$\tfrac1k\operatorname{tr}(a_{j'}^\dagger a_j\,\omega_{\mathrm{sym}})=\rho_{jj'}$.

**PPT across copy cuts.** For a cut $S=\{1,\dots,l\}$ introduce the branching isometry into a
two-block symmetric space,
$$W_l=(V_l\otimes V_{k-l})\,V^\dagger:\ \mathrm{Sym}^k\longrightarrow \mathrm{Sym}^l\otimes\mathrm{Sym}^{k-l},
\qquad W_lW_l^\dagger=\mathbb I.$$
Because the Dicke isometries are real, transposing the copies in $S$ commutes with symmetrising
within $S$, so on $\mathrm{Sym}^l\otimes\mathrm{Sym}^{k-l}$ the partial transpose $T_S$ acts as the
ordinary transpose of the first ($\mathrm{Sym}^l$) factor, denoted $\Gamma$. Since $W_l$ is an
isometry the two PSD conditions are equivalent:
$$\omega^{T_S}\succeq0\iff\big(W_l\,\omega_{\mathrm{sym}}\,W_l^\dagger\big)^{\Gamma}\succeq0,
\qquad l=1,\dots,\lfloor k/2\rfloor.$$

**Compressed SDP.**
$$\min_{\omega_{\mathrm{sym}}\succeq0}\ \operatorname{tr}(W_{\mathrm{obj}}\,\omega_{\mathrm{sym}})
\quad\text{s.t.}\quad
\tfrac1k\operatorname{tr}(a_{j'}^\dagger a_j\,\omega_{\mathrm{sym}})=\rho_{jj'},\ \
\big(W_l\,\omega_{\mathrm{sym}}\,W_l^\dagger\big)^{\Gamma}\succeq0\ \ \forall l\le\lfloor k/2\rfloor.$$
The $\Pi_{\mathrm{sym}}\omega\Pi_{\mathrm{sym}}=\omega$ constraint is dropped — it is automatic once the
variable lives on $\mathrm{Sym}^k$.


In [30]:
from math import comb

def dim_sym_kd(k: int, d: int):
    return comb(k + d - 1, d - 1)

def unrank(index:int, items:int, choose:int):
    """Unrank index in combinatorial number system: choose items from range [0, items-1]."""
    result = []
    for i in range(choose, 0, -1):
        x = items - 1
        while comb(x, i) > index:
            x -= 1
        result.append(x)
        index -= comb(x, i)
    return result

def occupation_nu(index: int, k:int, d:int):
    "Give the nu vector corresponding to that index"
    n_stars = k
    bars    = d - 1
    total   = n_stars + bars
    bar_positions = unrank(index, total, bars)
    bars_index = [total] + bar_positions + [-1] 
    stars = [bars_index[i] - 1 - bars_index[i + 1] for i in range(bars + 1)]
    return stars

def rank(bar_positions, items):
    """Rank a descending list of bar positions in combinatorial number system."""
    index = 0
    for i, x in enumerate(bar_positions):
        choose = len(bar_positions) - i  # counts down from bars to 1
        index += comb(x, choose)
    return index

def occupation_index(stars, k, d):
    "Give the index corresponding to the nu vector"
    total   = k + d - 1  # = n_stars + bars
    
    bars_index = [total]
    for s in stars[:-1]:
        bars_index.append(bars_index[-1] - 1 - s)
    
    bar_positions = bars_index[1:]
    return rank(bar_positions, total)

assert tuple([occupation_index(occupation_nu(i, 6,3), 6, 3) for i in range(dim_sym_kd(6,3))]) == tuple(range(dim_sym_kd(6,3)))

In [31]:
def nu_iterator(k: int, d: int):
    "iterate over all vector nu in C^d^k"
    ret = [0] * d

    def rec(bars_added, stars_remaining):
        if stars_remaining == 0:
            yield ret[:]
            return
        if bars_added >= d:
            return
        # place a star in current bin and recurse
        ret[bars_added] += 1
        yield from rec(bars_added, stars_remaining - 1)
        ret[bars_added] -= 1

        # move to next bin
        yield from rec(bars_added + 1, stars_remaining)
    yield from rec(0, k)



k, d = 3,4
for v in nu_iterator(k, d):
    print(v, end=", ")        
print()

def nu_set_iterator(nu: list[int]):
    "iterate over all the basis vector corresponding to nu"
    d = len(nu)
    k = sum(nu)
    def rec(j: int, ret):
        if(sum(nu) == 0):
            yield ret
        for i in range(len(nu)):
            if(nu[i] == 0):
                continue
            ret_i = ret + d**(k-j-1) * i
            nu[i] -= 1
            yield from rec(j+1, ret_i)
            nu[i] += 1 
    yield from rec(0, 0)

k, d = 3,4
for v in nu_set_iterator([1,1,1]):
    print(v, end=", ")        
print()


[3, 0, 0, 0], [2, 1, 0, 0], [2, 0, 1, 0], [2, 0, 0, 1], [1, 2, 0, 0], [1, 1, 1, 0], [1, 1, 0, 1], [1, 0, 2, 0], [1, 0, 1, 1], [1, 0, 0, 2], [0, 3, 0, 0], [0, 2, 1, 0], [0, 2, 0, 1], [0, 1, 2, 0], [0, 1, 1, 1], [0, 1, 0, 2], [0, 0, 3, 0], [0, 0, 2, 1], [0, 0, 1, 2], [0, 0, 0, 3], 
5, 7, 11, 15, 19, 21, 


In [32]:
from math import factorial, prod

def multinomial(k, nu):
    return factorial(k) // prod(factorial(n) for n in nu)

With the code above we can define the basis of $Sym({\mathbb{C}^d}^{\otimes k})$ in terms of its index. 
Now to build the matrix V with dimensions $|Sym({\mathbb{C}^d}^k)| \times |{\mathbb{C}^d}^k| = \binom{k + d -1}{k} \times d^k $

$$\begin{align*}
    V &= \sum_{i_1,i_2 \dots i_k}^d \frac{1}{\sqrt{\binom{k}{\nu(i_1,i_2 \dots i_k)}}}\ket{\nu(i_1,i_2 \dots i_k)}\bra{i_1,i_2 \dots i_k} \\
    &= \sum_{\nu} \frac{1}{\sqrt{\binom{k}{\nu(i_1,i_2 \dots i_k)}}} \sum_{\{i_1,i_2 \dots i_k\} \in \nu} \ket{\nu(i_1,i_2 \dots i_k)}\bra{i_1,i_2 \dots i_k} \\
\end{align*}$$

In [ ]:
import numpy as np

def V_builder(k:int,d:int):
    V = np.zeros((math.comb(k + d - 1, k), d**k))
    for nu in nu_iterator(k,d):
        fact = 1.0 / math.sqrt(multinomial(k, nu))
        nu_index = occupation_index(nu, k, d)
        V[nu_index, list(nu_set_iterator(nu))] = fact
    return V    

print(V_builder(3, 2))

[3, 0]
[2, 1]
[1, 2]
[0, 3]
[[1.         0.         0.         0.         0.         0.
  0.         0.        ]
 [0.         0.57735027 0.57735027 0.         0.57735027 0.
  0.         0.        ]
 [0.         0.         0.         0.57735027 0.         0.57735027
  0.57735027 0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         1.        ]]


Let's also make a function to compute $V_l \otimes V_{k-l}$ and $W_l = (V_l \otimes V_{k-l})V^\dagger$

In [34]:
def V_l_V_kl_builder(k:int,d:int,l:int):
    Vl = V_builder(l,d)
    Vkl = V_builder(k-l,d)
    return np.kron(Vl, Vkl)

def W_l_builder(k:int,d:int,l:int):
    return V_l_V_kl_builder(k,d,l) @ V_builder(k,d).transpose()

Let us construct $\Pi^\lambda_{A_{1 \dots k}} \otimes\mathbb{I}_{B_{1 \dots k}}$ and permute its subsystems such that it acts on $A_1B_1 A_2B_2 \dots A_kB_k$ instead of $A_1A_2\dots A_k B_1 B_2 \dots B_k$. 

Assuming $|\psi_{AB}\rangle_i \in \mathbb{C}^m \otimes \mathbb{C}^n \cong \mathbb{C}^d$

In [ ]:
def isotopyc_I_perm(m:int, n:int, k:int):

    ret = []

    def recA(new_index:int, dim_num:int):
        if dim_num >= k + 1:
            recB(new_index, 1)
            return
        
        stride = (n*m)**(k-dim_num) * n
        for i in range(new_index, new_index + m * stride, stride):
            recA(i, dim_num + 1)
    
    def recB(new_index:int, dim_num:int):
    
        if dim_num >= k + 1:
            ret.append(new_index)
            return
        
        stride = (n*m)**(k-dim_num)
        for i in range(new_index, new_index + n*stride, stride):
            recB(i, dim_num + 1)

    recA(0, 1)
    return ret

def isotypic_ot_I(m, n, k, lam):
    Pi_lambda = isotypic_projector(lam, m, k)
    I = np.identity(n**k)
    M = np.kron(Pi_lambda, I)            # ordered A_1..A_k B_1..B_k

    perm = np.asarray(isotopyc_I_perm(m, n, k))
    inv  = np.argsort(perm)
    return M[np.ix_(inv, inv)]           # now ordered A_1 B_1 ... A_k B_k



In [36]:
def alpha_dag_j_builder(k:int, d:int, j:int):
    alpha_dag_j = np.zeros((dim_sym_kd(k,d), dim_sym_kd(k-1,d)))
    for nu_darrow in nu_iterator(k-1,d):
        index_V_darrow = occupation_index(nu_darrow, k-1, d)
        nu_darrow[j] += 1
        index_V_darrow_e_j = occupation_index(nu_darrow, k, d)
        alpha_dag_j[index_V_darrow_e_j,index_V_darrow] = math.sqrt(nu_darrow[j])
    return alpha_dag_j

print(alpha_dag_j_builder(3,3,2))

[[0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.        ]
 [1.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.        ]
 [0.         1.         0.         0.         0.         0.        ]
 [0.         0.         1.41421356 0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         1.         0.         0.        ]
 [0.         0.         0.         0.         1.41421356 0.        ]
 [0.         0.         0.         0.         0.         1.73205081]]


Now we can go on to implement the SDP on the symmetric subspace.

In [ ]:
import picos

def SDP(lam: list[int], rho: np.ndarray, m: int, n: int, solver="cvxopt"):
    k = sum(lam)
    d, _ = rho.shape
    assert d == m * n

    sym_d: int   = dim_sym_kd(k, d)
    V   = V_builder(k, d)
    a_dag = [alpha_dag_j_builder(k, d, j) for j in range(d)]
    AdA = [[picos.Constant(Ad @ A.T) for A in a_dag] for Ad in a_dag]
    VPi = picos.Constant(V @ isotypic_ot_I(m, n, k, lam) @ V.T)
    Wls = [(picos.Constant(W_l_builder(k, d, l)), dim_sym_kd(l, d), dim_sym_kd(k - l, d))
       for l in range(1, k//2 + 1)]


    P = picos.Problem()
    omega_sym = picos.HermitianVariable("omega_sym", sym_d)

    # objective:  min tr( (V Pi^lam(x)I V^dag) omega_sym )
    P.set_objective("min", picos.trace(VPi * omega_sym).real) # type: ignore

    # (1) PSD
    P.add_constraint(omega_sym >> 0)

    #(2) marginal:  tr_{!=1}(omega)_{j,j'} = (1/k) tr( a_{j'}^dag a_j  omega ) = rho
    marg = picos.block([[ (1/k) * picos.trace(AdA[jp][j] * omega_sym) for jp in range(d)] for j in range(d)]) # type: ignore
    P.add_constraint(marg == picos.Constant("rho", rho))

    # (3) PPT on the l | k-l cuts,  l = 1 .. floor(k/2)   [your range(k//2) dropped the top cut]
    for Wl, dl, dkl in Wls:
        B = Wl * omega_sym * Wl.T
        P.add_constraint(B.partial_transpose(subsystems=0, dimensions=(dl, dkl)) >> 0)  # type: ignore

    P.solve(solver=solver)
    return P.value, omega_sym.value
